# Create dataset with PACE

## function to read in the icechunk

In [1]:
import earthaccess
import icechunk as ic
import xarray as xr

def create_ds(
    product="PACE_OCI_L3M_RRS",
    group="daily/0p1deg",
):
    url = f"https://data.source.coop/fish-pace/pace-oci/inregion/{product}"
    
    storage = ic.http_storage(url)
    
    auth = earthaccess.login()
    s3_credentials = auth.get_s3_credentials(daac="OBDAAC")
    
    url_prefix = "s3://ob-cumulus-prod-public/"
    
    virtual_creds = ic.credentials.containers_credentials({
        url_prefix: ic.credentials.s3_credentials(
            access_key_id=s3_credentials["accessKeyId"],
            secret_access_key=s3_credentials["secretAccessKey"],
            session_token=s3_credentials["sessionToken"],
        )
    })
    
    store = repo = ic.Repository.open(
        storage,
        authorize_virtual_chunk_access=virtual_creds,
    ).readonly_session("main").store

    ds = xr.open_zarr(
        store,
        consolidated=False,
        group=group,
        chunks={}
    )

    return ds

In [ ]:
%%script false --no-raise
# ug what a memory hog the concat is; For now just use  chunks_512
ds512 = create_ds("PACE_OCI_L3M_CHL", "daily/0p1deg/chunks_512")
ds16 = create_ds("PACE_OCI_L3M_CHL", "daily/0p1deg/chunks_16")

ds = xr.concat(
    [ds512, ds16],
    dim="time",
    coords="minimal",
    compat="override",
    combine_attrs="override",
).sortby("time")

ds

## The Arabian Ocean batch -- Daily

In [2]:
ds512 = create_ds("PACE_OCI_L3M_CHL", "daily/0p1deg/chunks_512")

# Define bounding box
lat_min, lat_max = 5, 31
lon_min, lon_max = 42, 80

zarr_ds = ds512.sel(
    lat=slice(lat_max, lat_min), 
    lon=slice(lon_min,lon_max)).drop_vars("palette")
zarr_ds

<xarray.Dataset> Size: 270MB
Dimensions:  (time: 683, lat: 260, lon: 380)
Coordinates:
  * time     (time) datetime64[ns] 5kB 2024-03-05 2024-03-06 ... 2026-01-31
  * lat      (lat) float32 1kB 30.95 30.85 30.75 30.65 ... 5.35 5.25 5.15 5.05
  * lon      (lon) float32 2kB 42.05 42.15 42.25 42.35 ... 79.75 79.85 79.95
Data variables:
    chlor_a  (time, lat, lon) float32 270MB dask.array<chunksize=(1, 260, 380), meta=np.ndarray>
Attributes: (12/64)
    product_name:                      PACE_OCI.20260131.L3m.DAY.CHL.V3_1.chl...
    instrument:                        OCI
    title:                             OCI Level-3 Standard Mapped Image
    project:                           Ocean Biology Processing Group (NASA/G...
    platform:                          PACE
    source:                            satellite observations from OCI-PACE
    ...                                ...
    identifier_product_doi:            10.5067/PACE/OCI/L3M/CHL/3.1
    keywords:                          Earth Science > Oceans > Ocean Chemist...
    keywords_vocabulary:               NASA Global Change Master Directory (G...
    data_bins:                         901994
    data_minimum:                      0.0030028503388166428
    data_maximum:                      98.65338897705078

In [5]:
import hvplot.xarray
import geoviews as gv
import geoviews.feature as gf
import cartopy.crs as ccrs
import panel as pn

var="chlor_a"

gv.extension("bokeh")
pn.extension()

img = zarr_ds[var].hvplot.image(
    x="lon",
    y="lat",
    groupby="time",
    geo=True,
    crs=ccrs.PlateCarree(),
    projection=ccrs.PlateCarree(),
    rasterize=True,
    dynamic=True,
    width=750,
    height=550,
    cmap="viridis",
    clim=(0, 2),
)

plot = gf.land * img * gf.coastline

pn.panel(
    plot,
    widget_location="bottom",
    widget_type="scrubber",
)

Column
    [0] HoloViews(DynamicMap, height=550, sizing_mode='fixed', widget_location='bottom', widget_type='scrubber', width=750)
    [1] WidgetBox(align=('center', 'end'))
        [0] Player(end=682, width=550)

In [ ]:
# this takes forever
import holoviews as hv
import geoviews as gv
import geoviews.feature as gf
import cartopy.crs as ccrs

gv.extension("bokeh")

img = da.hvplot.image(
    x="lon",
    y="lat",
    groupby="time",
    geo=True,
    crs=ccrs.PlateCarree(),
    projection=ccrs.PlateCarree(),
    rasterize=False,   # important
    dynamic=False,     # important for gif
    width=750,
    height=550,
    cmap="viridis",
    clim=(0, 2),
)

plot = gf.land * img * gf.coastline

hv.save(
    plot,
    "my_animation.gif",
    fmt="gif",
    fps=10,
)

## 8Day

Maybe this is better...

In [6]:
ds = create_ds("PACE_OCI_L3M_CHL", "8Day/0p1deg/chunks_512")

# Define bounding box
lat_min, lat_max = 5, 31
lon_min, lon_max = 42, 80

zarr_ds = ds.sel(
    lat=slice(lat_max, lat_min), 
    lon=slice(lon_min,lon_max)).drop_vars("palette")
zarr_ds

<xarray.Dataset> Size: 34MB
Dimensions:  (time: 87, lat: 260, lon: 380)
Coordinates:
  * time     (time) datetime64[ns] 696B 2024-03-05 2024-03-13 ... 2026-01-17
  * lat      (lat) float32 1kB 30.95 30.85 30.75 30.65 ... 5.35 5.25 5.15 5.05
  * lon      (lon) float32 2kB 42.05 42.15 42.25 42.35 ... 79.75 79.85 79.95
Data variables:
    chlor_a  (time, lat, lon) float32 34MB dask.array<chunksize=(1, 260, 380), meta=np.ndarray>
Attributes: (12/64)
    product_name:                      PACE_OCI.20260117_20260124.L3m.8D.CHL....
    instrument:                        OCI
    title:                             OCI Level-3 Standard Mapped Image
    project:                           Ocean Biology Processing Group (NASA/G...
    platform:                          PACE
    source:                            satellite observations from OCI-PACE
    ...                                ...
    identifier_product_doi:            10.5067/PACE/OCI/L3M/CHL/3.1
    keywords:                          Earth Science > Oceans > Ocean Chemist...
    keywords_vocabulary:               NASA Global Change Master Directory (G...
    data_bins:                         2705945
    data_minimum:                      0.0009999999310821295
    data_maximum:                      96.79078674316406

In [7]:
import hvplot.xarray
import geoviews as gv
import geoviews.feature as gf
import cartopy.crs as ccrs
import panel as pn

gv.extension("bokeh")
pn.extension()

img = zarr_ds[var].hvplot.image(
    x="lon",
    y="lat",
    groupby="time",
    geo=True,
    crs=ccrs.PlateCarree(),
    projection=ccrs.PlateCarree(),
    rasterize=True,
    dynamic=True,
    width=750,
    height=550,
    cmap="viridis",
    clim=(0, 2),
)

plot = gf.land * img * gf.coastline

pn.panel(
    plot,
    widget_location="bottom",
    widget_type="scrubber",
)

Column
    [0] HoloViews(DynamicMap, height=550, sizing_mode='fixed', widget_location='bottom', widget_type='scrubber', width=750)
    [1] WidgetBox(align=('center', 'end'))
        [0] Player(end=86, width=550)